# Use scikit-learn and custom library to predict temperature with `ibm-watsonx-ai`

This notebook contains steps and code to train a Scikit-Learn model that uses a custom defined transformer and use it with watsonx.ai Runtime service. Once the model is trained, this notebook contains steps to persist the model and custom defined transformer to watsonx.ai Runtime Repository, deploy and score it using watsonx.ai python client.

In this notebook, we use GNFUV dataset that contains mobile sensor readings data about humidity and temperature from Unmanned Surface Vehicles in a test-bed in Athens, to train a Scikit-Learn model for predicting the temperature. 

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goals

The learning goals of this notebook are:

- Train a model with custom defined transformer
- Persist the custom defined transformer and the model in watsonx.ai Runtime repository.
- Deploy the model using watsonx.ai Runtime Service
- Perform predictions using the deployed model

## Contents
1.	[Set up the environment](#1.-Set-up-the-environment)
2.	[Install python library containing custom transformer implementation](#2.-Install-the-library-containing-custom-transformer)
3.  [Prepare training data](#3.-Download-training-dataset-and-prepare-training-data)
4.	[Train the scikit-learn model](#4.-Train-a-model)
5.	[Save the model and library to watsonx.ai Runtime repository](#5.-Persist-the-model-and-custom-library)
6.	[Deploy and score data in the IBM Cloud](#6.-Deploy-and-Score)
7.  [Clean up](#7.-Clean-up)
8.	[Summary and next steps](#8.-Summary)


<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install and import the `ibm-watsonx-ai` and dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U setuptools | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Install-the-library-containing-custom-transformer"></a>

## 2. Install the library containing custom transformer

Library - `linalgnorm-0.1` is a python distributable package that contains the implementation of a user defined Scikit-Learn transformer - `LNormalizer` . <br>
Any 3rd party libraries that are required for the custom transformer must be defined as the dependency for the corresponding library that contains implementation of the transformer. 


In this section, we will create the library and install it in the current notebook environment. 

In [6]:
!mkdir -p linalgnorm-0.1/linalg_norm

Define a custom scikit transformer.

In [7]:
%%writefile linalgnorm-0.1/linalg_norm/sklearn_transformers.py

from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class LNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self, norm_ord=2):
        self.norm_ord = norm_ord
        self.row_norm_vals = None

    def fit(self, X, y=None):
        self.row_norm_vals = np.linalg.norm(X, ord=self.norm_ord, axis=0)

    def transform(self, X, y=None):
        return X / self.row_norm_vals

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        return self.transform(X, y)

    def get_norm_vals(self):
        return self.row_norm_vals

Writing linalgnorm-0.1/linalg_norm/sklearn_transformers.py


Wrap created code into Python source distribution package.

In [8]:
%%writefile linalgnorm-0.1/linalg_norm/__init__.py

__version__ = "0.1"

Writing linalgnorm-0.1/linalg_norm/__init__.py


In [9]:
%%writefile linalgnorm-0.1/README.md

A simple library containing a simple custom scikit estimator.

Writing linalgnorm-0.1/README.md


In [10]:
%%writefile linalgnorm-0.1/setup.py

from setuptools import setup

VERSION='0.1'
setup(
    name='linalgnorm',
    version=VERSION,
    url='https://github.ibm.com/NGP-TWC/repository/',
    author='IBM',
    author_email='ibm@ibm.com',
    license='IBM',
    packages=['linalg_norm'],
    zip_safe=False
)

Writing linalgnorm-0.1/setup.py


In [11]:
%%bash

cd linalgnorm-0.1
python setup.py sdist --formats=zip
cd ..
mv linalgnorm-0.1/dist/linalgnorm-0.1.zip .
rm -rf linalgnorm-0.1

Install the downloaded library using `pip` command

In [12]:
%pip install linalgnorm-0.1.zip

<a id="3.-Download-training-dataset-and-prepare-training-data"></a>

## 3. Download training dataset and prepare training data

Download the data from UCI repository - https://archive.ics.uci.edu/ml/machine-learning-databases/00452/GNFUV%20USV%20Dataset.zip

In [13]:
!rm -rf dataset
!mkdir dataset

In [14]:
import wget

wget.download(
    url="https://archive.ics.uci.edu/ml/machine-learning-databases/00452/GNFUV%20USV%20Dataset.zip",
    out="dataset/gnfuv_dataset.zip",
)

'dataset/gnfuv_dataset.zip'

In [15]:
!unzip dataset/gnfuv_dataset.zip -d dataset

Archive:  dataset/gnfuv_dataset.zip
  inflating: dataset/pi2/gnfuv-temp-exp1-55d487b85b-5g2xh_1.0.csv  
  inflating: dataset/pi3/gnfuv-temp-exp1-55d487b85b-2bl8b_1.0.csv  
  inflating: dataset/pi4/gnfuv-temp-exp1-55d487b85b-xcl97_1.0.csv  
  inflating: dataset/pi5/gnfuv-temp-exp1-55d487b85b-5ztk8_1.0.csv  
  inflating: dataset/README.pdf      


Create pandas datafame based on the downloaded dataset

In [16]:
import json
import os
from datetime import datetime
from json import JSONDecodeError

import numpy as np
import pandas as pd

In [17]:
home_dir = "./dataset"
pi_dirs = os.listdir(home_dir)

data_list = []
base_time = None
columns = None

for pi_dir in pi_dirs:
    if "pi" not in pi_dir:
        continue
    curr_dir = os.path.join(home_dir, pi_dir)
    data_file = os.path.join(curr_dir, os.listdir(curr_dir)[0])
    with open(data_file, "r") as f:
        line = f.readline().strip().replace("'", '"')
        while line != "":
            try:
                input_json = json.loads(line)
                sensor_datetime = datetime.fromtimestamp(input_json["time"])
                if base_time is None:
                    base_time = datetime(
                        sensor_datetime.year,
                        sensor_datetime.month,
                        sensor_datetime.day,
                        0,
                        0,
                        0,
                        0,
                    )
                input_json["time"] = (sensor_datetime - base_time).seconds
                data_list.append(list(input_json.values()))
                if columns is None:
                    columns = list(input_json.keys())
            except JSONDecodeError as je:
                pass
            line = f.readline().strip().replace("'", '"')

data_df = pd.DataFrame(data_list, columns=columns)

In [18]:
data_df.head()

,device,humidity,temperature,experiment,time
0,gnfuv-temp-exp1-55d487b85b-5g2xh,21.0,40.0,1.0,69557
1,gnfuv-temp-exp1-55d487b85b-5g2xh,21.0,40.0,1.0,69571
2,gnfuv-temp-exp1-55d487b85b-5g2xh,21.0,40.0,1.0,69577
3,gnfuv-temp-exp1-55d487b85b-5g2xh,21.0,40.0,1.0,69583
4,gnfuv-temp-exp1-55d487b85b-5g2xh,22.0,40.0,1.0,69589


Create training and test datasets from the downloaded GNFUV-USV dataset.

In [19]:
from sklearn.model_selection import train_test_split

y = data_df["temperature"]
X = data_df.drop("temperature", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=143
)

<a id="4.-Train-a-model"></a>

## 4. Train a model

In this section, you will use the custom transformer as a stage in the Scikit-Learn `Pipeline` and train a model.

#### Import the custom transformer 
Here, import the custom transformer that has been defined in `linalgnorm-0.1.zip` and create an instance of it that will inturn be used as stage in `sklearn.Pipeline`

In [20]:
from linalg_norm.sklearn_transformers import LNormalizer

In [21]:
lnorm_transf = LNormalizer()

Import other objects required to train a model

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

Now, you can create a `Pipeline` with user defined transformer as one of the stages and train the model

In [23]:
skl_pipeline = Pipeline(
    steps=[("normalizer", lnorm_transf), ("regression_estimator", LinearRegression())]
)
skl_pipeline.fit(X_train.loc[:, ["time", "humidity"]].values, y_train)

Pipeline(steps=[('normalizer', LNormalizer()),
                ('regression_estimator', LinearRegression())])

In [24]:
y_pred = skl_pipeline.predict(X_test.loc[:, ["time", "humidity"]].values)
rmse = np.mean((np.round(y_pred) - y_test.values) ** 2) ** 0.5
print("RMSE: {}".format(rmse))

RMSE: 2.213758431322581


<a id="5.-Persist-the-model-and-custom-library"></a>

## 5. Persist the model and custom library

In this section, using `ibm-watsonx-ai` SDK, you will ...
- save the library `linalgnorm-0.1.zip` in watsonx.ai Runtime repository by creating a package extension resource
- create a Software Specification resource and bind the package resource to it. This Software Specification resource will be used to configure the online deployment runtime environment for a model 
- bind Software Specification resource to the model and save the model to watsonx.ai Runtime repository

### Create package extension

Define the meta data required to create package extension resource. <br>

The value for `file_path` in `client.package_extensions.LibraryMetaNames.store()` contains the library file name that must be uploaded to the watsonx.ai Runtime.

**Note:** You can also use conda environment configuration file `yaml` as package extension input. In such case set the `TYPE` to `conda_yml` and `file_path` to yaml file.
```
client.package_extensions.ConfigurationMetaNames.TYPE = "conda_yml"
```

In [25]:
meta_prop_pkg_extn = {
    client.package_extensions.ConfigurationMetaNames.NAME: "K_Linag_norm_skl",
    client.package_extensions.ConfigurationMetaNames.DESCRIPTION: "Pkg extension for custom lib",
    client.package_extensions.ConfigurationMetaNames.TYPE: "pip_zip",
}

pkg_extn_details = client.package_extensions.store(
    meta_props=meta_prop_pkg_extn, file_path="linalgnorm-0.1.zip"
)
pkg_extn_id = client.package_extensions.get_id(pkg_extn_details)
pkg_extn_url = client.package_extensions.get_href(pkg_extn_details)

Creating package extension
SUCCESS


Display the details of the package extension resource that was created in the above cell.

In [26]:
details = client.package_extensions.get_details(pkg_extn_id)

### Create software specification and add custom library

Define the meta data required to create software spec resource and bind the package. This software spec resource will be used to configure the online deployment runtime environment for a model.

In [27]:
client.software_specifications.ConfigurationMetaNames.show()

---------------------------  ----  --------  --------------------------------
META_PROP NAME               TYPE  REQUIRED  SCHEMA
NAME                         str   Y
DESCRIPTION                  str   N
PACKAGE_EXTENSIONS           list  N
SOFTWARE_CONFIGURATION       dict  N         {'platform(required)': 'string'}
BASE_SOFTWARE_SPECIFICATION  dict  Y
---------------------------  ----  --------  --------------------------------


#### List base software specifications

In [28]:
client.software_specifications.list()

,NAME,ID,TYPE,STATE,REPLACEMENT
0,runtime-25.1-py3.12-xc,0144ea5e-9ff3-579a-9888-e252f75d24fa,base,supported,
1,watsonx-cfm-caikit-1.0,0cee3c55-472f-57b1-84bd-72f5d066dbe4,base,supported,watsonx-cfm-caikit-1.1
2,watsonx-textgen-fm-1.0,129aec82-7e65-5c78-b812-4c0a74b916f5,base,supported,
3,masking-flows-spark,13666829-5570-53a7-927b-52d42a101d93,base,supported,
4,runtime-25.1-r4.4,2b2ddb1d-6c0f-5b95-9d04-3056270ea5c6,base,supported,
5,tensorflow_rt24.1-py3.11,2c33167d-b11c-5490-a305-3e5e95db5c4d,base,supported,
6,pytorch-onnx_rt24.1-py3.11,2da185aa-eac3-59a5-bb8e-0e5b60458a15,base,supported,
7,runtime-25.1-py3.12-cuda,347e3aa2-a7f2-519c-a444-ade0e3433b74,base,supported,
8,onnxruntime_opset_19,368d2795-aaa7-59a0-834c-248c64a5a99e,base,supported,
9,runtime-24.1-py3.11-xc,36e20eca-b269-5e53-8f76-559f5cf898f2,base,supported,


#### Select base software specification to extend

In [29]:
base_sw_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")

#### Define new software specification based on base one and custom library

In [30]:
meta_prop_sw_spec = {
    client.software_specifications.ConfigurationMetaNames.NAME: "linalgnorm-0.1",
    client.software_specifications.ConfigurationMetaNames.DESCRIPTION: "Software specification for linalgnorm-0.1",
    client.software_specifications.ConfigurationMetaNames.BASE_SOFTWARE_SPECIFICATION: {
        "guid": base_sw_spec_id
    },
}

sw_spec_details = client.software_specifications.store(meta_props=meta_prop_sw_spec)
sw_spec_id = client.software_specifications.get_id(sw_spec_details)


client.software_specifications.add_package_extension(sw_spec_id, pkg_extn_id)

SUCCESS


'SUCCESS'

### Save the model

Define the metadata to save the trained model to watsonx.ai Runtime repository along with the information about the software spec resource required for the model. 

The `client.repository.ModelMetaNames.SOFTWARE_SPEC_ID` metadata property is used to specify the GUID of the software spec resource that needs to be associated with the model.

In [31]:
model_props = {
    client.repository.ModelMetaNames.NAME: "Temp prediction model with custom lib",
    client.repository.ModelMetaNames.TYPE: "scikit-learn_1.6",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}

Save the model to the watsonx.ai Runtime repository and display its saved metadata. 

In [32]:
published_model = client.repository.store_model(
    model=skl_pipeline, meta_props=model_props
)

In [33]:
published_model_id = client.repository.get_model_id(published_model)
model_details = client.repository.get_details(published_model_id)
print(json.dumps(model_details, indent=2))

{
  "metadata": {
    "name": "Temp prediction model with custom lib",
    "space_id": "fb3d528a-bf16-460e-bcf6-06f05d8ba57c",
    "resource_key": "5d21df9c-2621-4860-bf8c-8fc1b3ca17e1",
    "id": "348d592d-92f4-4e8e-9c79-c427ea03c9de",
    "created_at": "2026-01-15T09:53:31Z",
    "rov": {
      "member_roles": {
        "IBMid-696000GJGB": {
          "user_iam_id": "IBMid-696000GJGB",
          "roles": [
            "OWNER"
          ]
        }
      }
    },
    "owner": "IBMid-696000GJGB"
  },
  "entity": {
    "software_spec": {
      "id": "1247f51c-cc0d-49fe-b694-bbff242551ac"
    },
    "type": "scikit-learn_1.6"
  }
}


<a id="6.-Deploy-and-Score"></a>

## 6. Deploy and Score

In this section, you will deploy the saved model that uses the custom transformer and perform predictions. You will use watsonx.ai client to perform these tasks.

### Deploy the model

In [34]:
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "Deployment of custom lib model",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

created_deployment = client.deployments.create(published_model_id, meta_props=metadata)



######################################################################################

Synchronous deployment creation for id: '348d592d-92f4-4e8e-9c79-c427ea03c9de' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
......
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='d6dfb6ad-1d30-4730-8005-727f4a080305'
-----------------------------------------------------------------------------------------------




### Predict using the deployed model

**Note**: Here we use deployment `id` saved in published_model object. In next section, we show how to retrive deployment url from watsonx.ai Runtime instance.

In [35]:
deployment_id = client.deployments.get_id(created_deployment)

Now you can print an online scoring endpoint. 

In [36]:
scoring_endpoint = client.deployments.get_scoring_href(created_deployment)
print(scoring_endpoint)

Prepare the payload for prediction. The payload contains the input records for which predictions has to be performed.

In [37]:
scoring_payload = {
    "input_data": [{"fields": ["time", "humidity"], "values": [[79863, 47]]}]
}

Execute the method to perform online predictions and display the prediction results

In [38]:
predictions = client.deployments.score(deployment_id, scoring_payload)

In [39]:
print(json.dumps(predictions, indent=2))

{
  "predictions": [
    {
      "fields": [
        "prediction"
      ],
      "values": [
        [
          14.629242312262988
        ]
      ]
    }
  ]
}


<a id="7.-Clean-up"></a>
## 7. Clean up

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="8.-Summary"></a>

## 8. Summary

You successfully completed this notebook! 
 
You learned how to use a scikit-learn model with custom transformer in watsonx.ai Runtime service to deploy and score.

Check out our _[Online Documentation](https://www.ibm.com/cloud/watson-studio/autoai)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

## Author

**Krishnamurthy Arthanarisamy**, is a senior technical lead in IBM Watson Machine Learning team. Krishna works on developing cloud services that caters to different stages of machine learning and deep learning modeling life cycle.

**Lukasz Cmielowski**, PhD, is a Software Architect and Data Scientist at IBM.

**Mateusz Szewczyk**, Software Engineer at watsonx.ai

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.